# REGRAS DE NEGÓCIO - SISTEMA STCP

Sistema de decisão inteligente para otimização de carregamento de autocarros elétricos.

**8 Regras implementadas:**
1. Limites de Potência
2. Prioridades de Carregamento  
3. Limites de Temperatura
4. Disponibilidade de Estações
5. Distância ao Depósito
6. Tempo Estimado de Carregamento
7. Balanceamento de Carga
8. Saúde da Bateria

---

## REGRA 1: Limites de Potência

**Objetivo**: Garantir que nenhuma estação excede a sua potência máxima.

**Como funciona:**
- Cada estação tem potência máxima de **150 kW**
- Bateria de cada autocarro tem capacidade de **350 kWh**
- Níveis críticos: **20%** (mínimo), **80%** (ideal)

**Resultado:** Identificar se os limites estão a ser respeitados.

In [0]:
# REGRA 1: Configuração de Limites
POTENCIA_MAXIMA_ESTACAO = 150.0  # kW
BATERIA_MAXIMA_AUTOCARRO = 350.0  # kWh
BATERIA_MINIMA_SEGURANCA = 20.0  # %
BATERIA_IDEAL = 80.0  # %

print("="*80)
print("REGRA 1: LIMITES DE POTÊNCIA")
print("="*80)
print(f"✓ Potência máxima por estação: {POTENCIA_MAXIMA_ESTACAO} kW")
print(f"✓ Capacidade bateria: {BATERIA_MAXIMA_AUTOCARRO} kWh")
print(f"✓ Nível crítico: {BATERIA_MINIMA_SEGURANCA}%")
print(f"✓ Nível ideal: {BATERIA_IDEAL}%")

# Verificar se estações respeitam limites
display(spark.sql(f"""
    SELECT 
        device_id,
        ROUND(currentPower, 1) as potencia_atual_kw,
        {POTENCIA_MAXIMA_ESTACAO} as limite_kw,
        CASE 
            WHEN currentPower > {POTENCIA_MAXIMA_ESTACAO} THEN '❌ EXCEDE LIMITE'
            WHEN currentPower > {POTENCIA_MAXIMA_ESTACAO * 0.9} THEN '⚠️ PRÓXIMO DO LIMITE'
            ELSE '✓ OK'
        END as status_potencia
    FROM (
        SELECT device_id, currentPower,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    ) WHERE rn = 1
    ORDER BY currentPower DESC
"""))

# Verificar se autocarros respeitam limites de bateria
display(spark.sql(f"""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        CASE 
            WHEN batteryLevel < {BATERIA_MINIMA_SEGURANCA} THEN '🔴 CRÍTICO'
            WHEN batteryLevel < 50 THEN '🟡 BAIXO'
            WHEN batteryLevel > {BATERIA_IDEAL} THEN '🟢 IDEAL/ALTO'
            ELSE '✓ OK'
        END as status_bateria
    FROM (
        SELECT device_id, batteryLevel,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    ) WHERE rn = 1
    ORDER BY batteryLevel ASC
"""))

REGRA 1: LIMITES DE POTÊNCIA
✓ Potência máxima por estação: 150.0 kW
✓ Capacidade bateria: 350.0 kWh
✓ Nível crítico: 20.0%
✓ Nível ideal: 80.0%


device_id,potencia_atual_kw,limite_kw,status_potencia
CS-001,80.0,150.0,✓ OK
CS-002,0.0,150.0,✓ OK
CS-003,0.0,150.0,✓ OK


device_id,bateria_percent,status_bateria
BUS-003,25.5,🟡 BAIXO
BUS-001,47.3,🟡 BAIXO
BUS-002,71.7,✓ OK


## REGRA 2: Prioridades de Carregamento

**Objetivo**: Decidir qual autocarro deve carregar primeiro com base em prioridades.

**Como funciona:**
- Cada rota tem uma **prioridade** (Crítica=40pts, Alta=30pts, Normal=20pts)
- Calcula-se um **score de urgência** baseado em:
  - Prioridade da rota (40%)
  - Nível de bateria atual (60%)
  
**Fórmula:**
```
Score = Pontos_Prioridade + Pontos_Bateria
- Se bateria < mínimo → +60 pontos
- Se bateria entre mínimo e (mínimo+10%) → +30 pontos
- Caso contrário → 0 pontos
```

**Resultado:** Lista ordenada de autocarros por urgência.

In [0]:
# REGRA 2: Criar tabela de prioridades (se ainda não existe)
spark.sql("DROP TABLE IF EXISTS route_priorities")

prioridades = spark.createDataFrame([
    ("BUS-001", "200", "Bolhão - Castelo Queijo", "Alta", 9.85, 30.0),
    ("BUS-002", "201", "Aliados - Viso", "Normal", 9.78, 30.0),
    ("BUS-003", "202", "Aliados - Passeio Alegre", "Crítica", 5.31, 20.0)
], ["device_id", "route_id", "route_name", "priority", "distance_km", "min_battery_required"])

prioridades.write.mode("overwrite").saveAsTable("route_priorities")

print("="*80)
print("REGRA 2: PRIORIDADES DE CARREGAMENTO")
print("="*80)

# Calcular score de urgência
display(spark.sql("""
    WITH latest_bus AS (
        SELECT device_id, batteryLevel,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    ),
    urgency_calc AS (
        SELECT 
            l.device_id,
            ROUND(l.batteryLevel, 1) as bateria_atual,
            r.route_id,
            r.route_name,
            r.priority,
            r.min_battery_required as bateria_minima,
            -- Pontos de prioridade
            CASE 
                WHEN r.priority = 'Crítica' THEN 40
                WHEN r.priority = 'Alta' THEN 30
                WHEN r.priority = 'Normal' THEN 20
                ELSE 10
            END as pontos_prioridade,
            -- Pontos de bateria
            CASE 
                WHEN l.batteryLevel < r.min_battery_required THEN 60
                WHEN l.batteryLevel < (r.min_battery_required + 10) THEN 30
                ELSE 0
            END as pontos_bateria
        FROM latest_bus l
        JOIN route_priorities r ON l.device_id = r.device_id
        WHERE l.rn = 1
    )
    SELECT 
        device_id,
        route_id,
        route_name,
        priority as prioridade,
        bateria_atual,
        bateria_minima,
        pontos_prioridade,
        pontos_bateria,
        (pontos_prioridade + pontos_bateria) as score_urgencia,
        CASE 
            WHEN (pontos_prioridade + pontos_bateria) >= 70 THEN '🔴 URGENTE'
            WHEN (pontos_prioridade + pontos_bateria) >= 50 THEN '🟡 PRIORITÁRIO'
            WHEN (pontos_prioridade + pontos_bateria) >= 30 THEN '🟢 NORMAL'
            ELSE '⚪ NÃO NECESSÁRIO'
        END as classificacao
    FROM urgency_calc
    ORDER BY score_urgencia DESC
"""))

REGRA 2: PRIORIDADES DE CARREGAMENTO


device_id,route_id,route_name,prioridade,bateria_atual,bateria_minima,pontos_prioridade,pontos_bateria,score_urgencia,classificacao
BUS-003,202,Aliados - Passeio Alegre,Crítica,25.5,20.0,40,30,70,🔴 URGENTE
BUS-001,200,Bolhão - Castelo Queijo,Alta,47.3,30.0,30,0,30,🟢 NORMAL
BUS-002,201,Aliados - Viso,Normal,71.7,30.0,20,0,20,⚪ NÃO NECESSÁRIO


## REGRA 3: Limites de Temperatura

**Objetivo**: Proteger baterias e carregadores de sobreaquecimento.

**Como funciona:**
- **Bateria**: 
  - Temperatura máxima: **45°C** → PARAR carregamento
  - Temperatura ideal: **15-35°C**
  - Temperatura baixa: **<15°C** → Pré-aquecer
  
- **Carregador**:
  - Temperatura máxima: **60°C** → Aguardar arrefecimento

**Ações automáticas:**
- Temperatura > 45°C → Potência = 0 kW
- Temperatura > 35°C → Potência = 80 kW  
- Temperatura < 15°C → Potência = 50 kW
- Temperatura ideal → Potência = 150 kW

In [0]:
# REGRA 3: Limites de Temperatura
TEMP_BATERIA_MAX = 45.0
TEMP_BATERIA_IDEAL_MIN = 15.0
TEMP_BATERIA_IDEAL_MAX = 35.0
TEMP_CARREGADOR_MAX = 60.0

print("="*80)
print("REGRA 3: LIMITES DE TEMPERATURA")
print("="*80)
print(f"✓ Bateria - Máximo: {TEMP_BATERIA_MAX}°C")
print(f"✓ Bateria - Ideal: {TEMP_BATERIA_IDEAL_MIN}°C - {TEMP_BATERIA_IDEAL_MAX}°C")
print(f"✓ Carregador - Máximo: {TEMP_CARREGADOR_MAX}°C")

# Analisar temperatura das baterias
print("\n--- TEMPERATURA DAS BATERIAS ---")
display(spark.sql(f"""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria,
        ROUND(batteryTemperature, 1) as temperatura_C,
        CASE 
            WHEN batteryTemperature > {TEMP_BATERIA_MAX} THEN '🔴 CRÍTICO'
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN '🟡 ALTO'
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN '🔵 FRIO'
            ELSE '✓ IDEAL'
        END as status_temp,
        CASE 
            WHEN batteryTemperature > {TEMP_BATERIA_MAX} THEN 'PARAR carregamento'
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN 'REDUZIR para 80 kW'
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN 'PRÉ-AQUECER (50 kW)'
            ELSE 'Potência normal (150 kW)'
        END as acao_recomendada,
        CASE 
            WHEN batteryTemperature > {TEMP_BATERIA_MAX} THEN 0.0
            WHEN batteryTemperature > {TEMP_BATERIA_IDEAL_MAX} THEN 80.0
            WHEN batteryTemperature < {TEMP_BATERIA_IDEAL_MIN} THEN 50.0
            ELSE 150.0
        END as potencia_ajustada_kw
    FROM (
        SELECT device_id, batteryLevel, batteryTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    ) WHERE rn = 1
    ORDER BY batteryTemperature DESC
"""))

# Analisar temperatura dos carregadores
print("\n--- TEMPERATURA DOS CARREGADORES ---")
display(spark.sql(f"""
    SELECT 
        device_id,
        state as estado,
        ROUND(chargerTemperature, 1) as temperatura_C,
        CASE 
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX} THEN '🔴 SOBREAQUECIMENTO'
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX * 0.9} THEN '🟡 PRÓXIMO DO LIMITE'
            ELSE '✓ NORMAL'
        END as status_temp,
        CASE 
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX} THEN 'AGUARDAR arrefecimento'
            WHEN chargerTemperature > {TEMP_CARREGADOR_MAX * 0.9} THEN 'MONITORIZAR'
            ELSE 'Operação normal'
        END as acao
    FROM (
        SELECT device_id, state, chargerTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    ) WHERE rn = 1
    ORDER BY chargerTemperature DESC
"""))

REGRA 3: LIMITES DE TEMPERATURA
✓ Bateria - Máximo: 45.0°C
✓ Bateria - Ideal: 15.0°C - 35.0°C
✓ Carregador - Máximo: 60.0°C

--- TEMPERATURA DAS BATERIAS ---


device_id,bateria,temperatura_C,status_temp,acao_recomendada,potencia_ajustada_kw
BUS-003,25.5,25.0,✓ IDEAL,Potência normal (150 kW),150.0
BUS-002,71.7,20.6,✓ IDEAL,Potência normal (150 kW),150.0
BUS-001,47.3,18.5,✓ IDEAL,Potência normal (150 kW),150.0



--- TEMPERATURA DOS CARREGADORES ---


device_id,estado,temperatura_C,status_temp,acao
CS-001,occupied,36.4,✓ NORMAL,Operação normal
CS-002,fault,28.0,✓ NORMAL,Operação normal
CS-003,available,22.0,✓ NORMAL,Operação normal


## REGRA 4: Disponibilidade de Estações

**Objetivo**: Verificar quais estações estão disponíveis para alocar aos autocarros.

**Como funciona:**
- **IDLE** + Temperatura < 60°C → DISPONÍVEL
- **CHARGING** → OCUPADO (aguardar)
- **FAULT** → EM MANUTENÇÃO (não alocar)
- Temperatura ≥ 60°C → ARREFECIMENTO NECESSÁRIO

**Estados possíveis:**
- IDLE (livre)
- CHARGING (a carregar)
- FAULT (avaria)
- MAINTENANCE (manutenção)

**Resultado:** Lista de estações e sua disponibilidade real.

In [0]:
print("="*80)
print("REGRA 4: DISPONIBILIDADE DE ESTAÇÕES")
print("="*80)

display(spark.sql("""
    WITH latest_chargers AS (
        SELECT 
            device_id,
            state,
            ROUND(currentPower, 1) as potencia_atual,
            connectedBus,
            ROUND(chargerTemperature, 1) as temperatura,
            ROUND(latitude, 6) as lat,
            ROUND(longitude, 6) as lon,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    )
    SELECT 
        device_id,
        state as estado,
        potencia_atual,
        connectedBus as autocarro_conectado,
        temperatura,
        CONCAT(lat, ', ', lon) as localizacao,
        CASE 
            WHEN state = 'IDLE' AND temperatura < 60.0 THEN '✅ DISPONÍVEL'
            WHEN state = 'CHARGING' THEN '🔵 OCUPADO'
            WHEN temperatura >= 60.0 THEN '🔴 ARREFECIMENTO'
            WHEN state = 'FAULT' THEN '⚠️ MANUTENÇÃO'
            ELSE '❌ INDISPONÍVEL'
        END as disponibilidade,
        CASE 
            WHEN state = 'IDLE' AND temperatura < 60.0 THEN 1
            ELSE 0
        END as pode_alocar
    FROM latest_chargers
    WHERE rn = 1
    ORDER BY pode_alocar DESC, device_id
"""))

# Resumo de disponibilidade
print("\n--- RESUMO ---")
display(spark.sql("""
    WITH latest_chargers AS (
        SELECT state, chargerTemperature,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    )
    SELECT 
        COUNT(*) as total_estacoes,
        SUM(CASE WHEN state = 'IDLE' AND chargerTemperature < 60.0 THEN 1 ELSE 0 END) as disponiveis,
        SUM(CASE WHEN state = 'CHARGING' THEN 1 ELSE 0 END) as ocupadas,
        SUM(CASE WHEN chargerTemperature >= 60.0 THEN 1 ELSE 0 END) as arrefecimento,
        SUM(CASE WHEN state = 'FAULT' THEN 1 ELSE 0 END) as em_manutencao
    FROM latest_chargers
    WHERE rn = 1
"""))

REGRA 4: DISPONIBILIDADE DE ESTAÇÕES


device_id,estado,potencia_atual,autocarro_conectado,temperatura,localizacao,disponibilidade,pode_alocar
CS-001,occupied,80.0,BUS-004,36.4,"41.152661, -8.579658",❌ INDISPONÍVEL,0
CS-002,fault,0.0,null,28.0,"41.152671, -8.579668",❌ INDISPONÍVEL,0
CS-003,available,0.0,null,22.0,"41.152681, -8.579678",❌ INDISPONÍVEL,0



--- RESUMO ---


total_estacoes,disponiveis,ocupadas,arrefecimento,em_manutencao
3,0,0,0,0


## REGRA 5: Distância ao Depósito

**Objetivo**: Calcular se cada autocarro consegue chegar ao depósito com a bateria atual.

**Como funciona:**
- Localização do depósito: **lat=41.183580, lon=-8.618978**
- Consumo médio: **1.5 kWh/km**
- Margem de segurança: **20%** (multiplicador 1.2)

**Cálculo:**
1. Calcular distância do autocarro ao depósito (fórmula Haversine simplificada)
2. Calcular autonomia atual = bateria_kWh / 1.5
3. Comparar: autonomia ≥ distância × 1.2?

**Classificação:**
- 🔴 CRÍTICO: Não chega (urgência 100)
- 🟡 ALERTA: Margem reduzida (urgência 60)
- ✅ OK: Consegue chegar (urgência 0)

In [0]:
# REGRA 5: Constantes
DEPOT_LAT = 41.183580
DEPOT_LON = -8.618978
CONSUMO_POR_KM = 1.5
MARGEM_SEGURANCA = 1.2

print("="*80)
print("REGRA 5: DISTÂNCIA À ESTAÇÃO CARREGAMENTO")
print("="*80)
print(f"✓ Estação Carregamento: ({DEPOT_LAT}, {DEPOT_LON})")
print(f"✓ Consumo: {CONSUMO_POR_KM} kWh/km")
print(f"✓ Margem: {MARGEM_SEGURANCA}x (20%)")

display(spark.sql(f"""
    WITH latest_bus AS (
        SELECT 
            device_id,
            ROUND(batteryLevel, 1) as bateria_percent,
            ROUND(batteryKwh, 1) as bateria_kwh,
            latitude,
            longitude,
            state,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    ),
    distance_calc AS (
        SELECT 
            device_id,
            bateria_percent,
            bateria_kwh,
            state,
            -- Distância aproximada (fórmula Haversine simplificada)
            ROUND(
                111.0 * SQRT(
                    POW(({DEPOT_LAT} - latitude), 2) + 
                    POW(({DEPOT_LON} - longitude) * COS(latitude * 3.14159 / 180), 2)
                ), 
                2
            ) as distancia_depot_km,
            -- Autonomia atual
            ROUND(bateria_kwh / {CONSUMO_POR_KM}, 1) as autonomia_km
        FROM latest_bus
        WHERE rn = 1
    )
    SELECT 
        device_id,
        bateria_percent,
        distancia_depot_km,
        autonomia_km,
        ROUND(distancia_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA}, 1) as energia_necessaria_kwh,
        ROUND(autonomia_km - (distancia_depot_km * {MARGEM_SEGURANCA}), 1) as margem_km,
        CASE 
            WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN '🔴 CRÍTICO - Não chega'
            WHEN autonomia_km < (distancia_depot_km * 1.5) THEN '🟡 ALERTA - Margem reduzida'
            ELSE '✅ OK - Consegue chegar'
        END as status_alcance,
        CASE 
            WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 100
            WHEN autonomia_km < (distancia_depot_km * 1.5) THEN 60
            ELSE 0
        END as urgencia_alcance
    FROM distance_calc
    ORDER BY urgencia_alcance DESC
"""))

REGRA 5: DISTÂNCIA À ESTAÇÃO CARREGAMENTO
✓ Estação Carregamento: (41.18358, -8.618978)
✓ Consumo: 1.5 kWh/km
✓ Margem: 1.2x (20%)


device_id,bateria_percent,distancia_depot_km,autonomia_km,energia_necessaria_kwh,margem_km,status_alcance,urgencia_alcance
BUS-001,47.3,5.79,110.3,10.4,103.4,✅ OK - Consegue chegar,0
BUS-002,71.7,3.64,167.2,6.6,162.8,✅ OK - Consegue chegar,0
BUS-003,25.5,5.86,59.5,10.5,52.5,✅ OK - Consegue chegar,0


## REGRA 6: Tempo Estimado de Carregamento

**Objetivo**: Calcular quanto tempo cada autocarro precisa para atingir bateria mínima.

**Como funciona:**
- Capacidade bateria: **350 kWh**
- Energia a carregar = (bateria_necessária% × 350) - bateria_atual_kWh
- Tempo = Energia / Potência

**Cenários de potência:**
- **150 kW** → Carregamento ultra-rápido
- **120 kW** → Carregamento rápido
- **80 kW** → Carregamento normal

**Classificação:**
- < 30 min → Carregamento rápido
- 30-60 min → Carregamento médio
- ">" 60 min → Carregamento longo

In [0]:
print("="*80)
print("REGRA 6: TEMPO ESTIMADO DE CARREGAMENTO")
print("="*80)

display(spark.sql("""
    WITH latest_bus AS (
        SELECT 
            b.device_id,
            ROUND(b.batteryLevel, 1) as bateria_atual_percent,
            ROUND(b.batteryKwh, 1) as bateria_atual_kwh,
            r.min_battery_required,
            ROW_NUMBER() OVER (PARTITION BY b.device_id ORDER BY b.event_time DESC) as rn
        FROM bus_telemetry b
        JOIN route_priorities r ON b.device_id = r.device_id
    ),
    charging_calc AS (
        SELECT 
            device_id,
            bateria_atual_percent,
            min_battery_required,
            ROUND((min_battery_required / 100.0) * 350.0 - bateria_atual_kwh, 1) as energia_a_carregar_kwh
        FROM latest_bus
        WHERE rn = 1
    )
    SELECT 
        device_id,
        bateria_atual_percent as bateria_atual,
        min_battery_required as bateria_necessaria,
        energia_a_carregar_kwh,
        -- Tempos por potência
        CAST(ROUND((energia_a_carregar_kwh / 150.0) * 60, 0) AS INT) as tempo_150kw_min,
        CAST(ROUND((energia_a_carregar_kwh / 120.0) * 60, 0) AS INT) as tempo_120kw_min,
        CAST(ROUND((energia_a_carregar_kwh / 80.0) * 60, 0) AS INT) as tempo_80kw_min,
        CASE 
            WHEN bateria_atual_percent >= min_battery_required THEN '✅ Já carregado'
            WHEN (energia_a_carregar_kwh / 150.0) <= 0.5 THEN '⚡ Rápido (<30min)'
            WHEN (energia_a_carregar_kwh / 150.0) <= 1.0 THEN '🔵 Médio (30-60min)'
            ELSE '🟡 Longo (>60min)'
        END as categoria_tempo
    FROM charging_calc
    WHERE energia_a_carregar_kwh > 0
    ORDER BY energia_a_carregar_kwh DESC
"""))

REGRA 6: TEMPO ESTIMADO DE CARREGAMENTO


device_id,bateria_atual,bateria_necessaria,energia_a_carregar_kwh,tempo_150kw_min,tempo_120kw_min,tempo_80kw_min,categoria_tempo


## REGRA 7: Balanceamento de Carga

**Objetivo**: Distribuir carga entre estações sem sobrecarregar a rede elétrica.

**Como funciona:**
- **Potência total disponível**: 450 kW (3 estações × 150 kW)
- **Limite da rede**: 400 kW (para não sobrecarregar)

**Estados da rede:**
- **>400 kW**: SOBRECARGA → Reduzir para 80% por estação
- **360-400 kW** (90-100%): ALERTA → Reduzir para 90%
- **280-360 kW** (70-90%): CARGA ELEVADA → Monitorizar
- **<280 kW** (<70%): NORMAL → Potência máxima

**Ação automática:** Ajustar potência de TODAS as estações proporcionalmente.

In [0]:
# REGRA 7: Constantes
POTENCIA_TOTAL_DISPONIVEL = 450.0
LIMITE_CARGA_REDE = 400.0

print("="*80)
print("REGRA 7: BALANCEAMENTO DE CARGA")
print("="*80)
print(f"✓ Potência total: {POTENCIA_TOTAL_DISPONIVEL} kW")
print(f"✓ Limite rede: {LIMITE_CARGA_REDE} kW")

display(spark.sql(f"""
    WITH latest_chargers AS (
        SELECT 
            device_id,
            currentPower,
            state,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    ),
    carga_total AS (
        SELECT 
            ROUND(SUM(currentPower), 1) as carga_atual_kw,
            COUNT(CASE WHEN state = 'CHARGING' THEN 1 END) as estacoes_ativas,
            COUNT(CASE WHEN state = 'IDLE' THEN 1 END) as estacoes_disponiveis
        FROM latest_chargers
        WHERE rn = 1
    )
    SELECT 
        carga_atual_kw,
        estacoes_ativas,
        estacoes_disponiveis,
        ROUND({POTENCIA_TOTAL_DISPONIVEL} - carga_atual_kw, 1) as potencia_livre_kw,
        ROUND({LIMITE_CARGA_REDE} - carga_atual_kw, 1) as margem_rede_kw,
        ROUND((carga_atual_kw / {LIMITE_CARGA_REDE}) * 100, 1) as percentagem_uso,
        CASE 
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE} THEN '🔴 SOBRECARGA'
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.9} THEN '🟡 ALERTA'
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.7} THEN '🟠 CARGA ELEVADA'
            ELSE '✅ NORMAL'
        END as status_rede,
        CASE 
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE} THEN ROUND((carga_atual_kw / GREATEST(estacoes_ativas, 1)) * 0.8, 1)
            WHEN carga_atual_kw > {LIMITE_CARGA_REDE * 0.9} THEN ROUND((carga_atual_kw / GREATEST(estacoes_ativas, 1)) * 0.9, 1)
            ELSE 150.0
        END as potencia_maxima_ajustada
    FROM carga_total
"""))

# Detalhe por estação
print("\n--- CONSUMO POR ESTAÇÃO ---")
display(spark.sql("""
    SELECT 
        device_id,
        state,
        ROUND(currentPower, 1) as potencia_kw,
        connectedBus,
        ROUND((currentPower / 150.0) * 100, 1) as percentagem_capacidade
    FROM (
        SELECT device_id, state, currentPower, connectedBus,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry
    ) WHERE rn = 1
    ORDER BY currentPower DESC
"""))

REGRA 7: BALANCEAMENTO DE CARGA
✓ Potência total: 450.0 kW
✓ Limite rede: 400.0 kW


carga_atual_kw,estacoes_ativas,estacoes_disponiveis,potencia_livre_kw,margem_rede_kw,percentagem_uso,status_rede,potencia_maxima_ajustada
80.0,0,0,370.0,320.0,20.0,✅ NORMAL,150.0



--- CONSUMO POR ESTAÇÃO ---


device_id,state,potencia_kw,connectedBus,percentagem_capacidade
CS-001,occupied,80.0,BUS-004,53.3
CS-002,fault,0.0,null,0.0
CS-003,available,0.0,null,0.0


## REGRA 8: Saúde da Bateria (Longevidade)

**Objetivo**: Preservar vida útil da bateria evitando descargas profundas e cargas excessivas.

**Como funciona:**
- **Ciclos de vida**: ~3000 ciclos esperados
- **Faixa ideal**: 20-80% (maximiza longevidade)

**Zonas de operação:**
- **<20%**: ⚠️ Descarga profunda (reduz vida útil) → Carregar até 30%
- **20-80%**: ✅ FAIXA IDEAL (boa para longevidade) → Carregar até 80%
- **>80%**: ⚠️ Sobrecarga (pode degradar) → NÃO carregar mais

**Impacto:**
- Operar sempre 20-80% → +30% vida útil
- Descargas <20% frequentes → -20% vida útil
- Cargas >80% frequentes → -15% vida útil

In [0]:
# REGRA 8: Constantes
CICLOS_MAX = 3000
FAIXA_IDEAL_MIN = 20.0
FAIXA_IDEAL_MAX = 80.0

print("="*80)
print("REGRA 8: SAÚDE DA BATERIA")
print("="*80)
print(f"✓ Ciclos esperados: {CICLOS_MAX}")
print(f"✓ Faixa ideal: {FAIXA_IDEAL_MIN}% - {FAIXA_IDEAL_MAX}%")

display(spark.sql(f"""
    WITH latest_bus AS (
        SELECT 
            device_id,
            ROUND(batteryLevel, 1) as bateria_percent,
            ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry
    )
    SELECT 
        device_id,
        bateria_percent,
        CASE 
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN '🔴 DESCARGA PROFUNDA'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN '🟡 SOBRECARGA'
            ELSE '✅ FAIXA IDEAL'
        END as zona_operacao,
        CASE 
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN 'Reduz vida útil (-20%)'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'Pode degradar (-15%)'
            ELSE 'Ótimo para longevidade (+30%)'
        END as impacto_longevidade,
        CASE 
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN {FAIXA_IDEAL_MIN + 10}
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN bateria_percent
            ELSE {FAIXA_IDEAL_MAX}
        END as nivel_alvo_carregamento,
        CASE 
            WHEN bateria_percent < {FAIXA_IDEAL_MIN} THEN 'Carregar até 30% (não 100%)'
            WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'NÃO carregar mais'
            ELSE 'Carregar até 80%'
        END as recomendacao
    FROM latest_bus
    WHERE rn = 1
    ORDER BY bateria_percent ASC
"""))

REGRA 8: SAÚDE DA BATERIA
✓ Ciclos esperados: 3000
✓ Faixa ideal: 20.0% - 80.0%


device_id,bateria_percent,zona_operacao,impacto_longevidade,nivel_alvo_carregamento,recomendacao
BUS-003,25.5,✅ FAIXA IDEAL,Ótimo para longevidade (+30%),80.0,Carregar até 80%
BUS-001,47.3,✅ FAIXA IDEAL,Ótimo para longevidade (+30%),80.0,Carregar até 80%
BUS-002,71.7,✅ FAIXA IDEAL,Ótimo para longevidade (+30%),80.0,Carregar até 80%
